# 01 - OpenCV Introduction

## From Pixels to Perception

**OpenCV** (Open Source Computer Vision Library) is the backbone of almost every CV application. Before AI can "see" anything, the image must be loaded, cleaned, and transformed — and OpenCV is the tool that does it.

In this notebook, we will build up from first principles:

| Section | What You'll Learn |
|---|---|
| **The Matrix** | Images are just NumPy arrays of numbers |
| **Loading & Cropping** | Read real images and extract regions of interest |
| **Colour Spaces** | Convert between BGR, RGB, Gray, and HSV |
| **Edge Detection** | Find outlines with Gaussian blur + Canny |
| **Live Webcam** | Stream video frames in a loop |
| **Real-Time FX** | Pencil sketch, HUD overlays, colour tracking |
| **Drawing App** | Track a coloured object and leave a trail |

> **Prerequisite:** A working webcam (built-in or USB). All code runs on CPU — no GPU needed.

> **How to exit:** Press `q` in any OpenCV window to close it safely.

---

## Welcome To The Matrix

Computers don't see colours; they see a massive grid of numbers. An image is just a 3D Matrix (or a NumPy Array) with dimensions: $Height \times Width \times Channels$.

Before we look at real data, let's create a $10 \times 10$ grid of random numbers representing a grayscale image and tell the computer to treat them as pixels.

In grayscale, each pixel is a single number from 0 (Black) to 255 (White).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

# --- Grayscale: a single channel, values 0 (black) → 255 (white) ---
random_gray = np.random.randint(0, 256, (10, 10), dtype="uint8")

print(f"Shape : {random_gray.shape}   (Height × Width)")
print(f"Dtype : {random_gray.dtype}")
print(f"Range : {random_gray.min()} – {random_gray.max()}\n")
print("The Computer's View (raw numbers):\n")
print(random_gray)

# Visualise — cmap='gray' maps 0→black, 255→white
print("\nThe Human's View:")
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(random_gray, cmap="gray", vmin=0, vmax=255)
fig.colorbar(im, ax=ax, label="Pixel intensity")
ax.set_title("10 × 10 Random Grayscale Image")
plt.tight_layout()
plt.show()

To get colour, we need three layers: Blue, Green, and Red. This is why a colour image is a 3D array. The "depth" of the matrix is 3.

In [ ]:
# --- Colour: three channels (Red, Green, Blue) stacked in depth ---
random_color = np.random.randint(0, 256, (10, 10, 3), dtype="uint8")

print(f"Shape : {random_color.shape}   (Height × Width × Channels)")
print(f"Total values per pixel: {random_color.shape[2]}  (one per colour channel)\n")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Show the combined RGB image
axes[0].imshow(random_color)
axes[0].set_title("Combined (RGB)")

# Show each channel separately
channel_names = ["Red", "Green", "Blue"]
cmaps = ["Reds", "Greens", "Blues"]
for i, (name, cmap) in enumerate(zip(channel_names, cmaps)):
    axes[i + 1].imshow(random_color[:, :, i], cmap=cmap, vmin=0, vmax=255)
    axes[i + 1].set_title(f"{name} Channel")

for ax in axes:
    ax.axis("off")

plt.suptitle("One Image = Three Layers of Numbers", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Loading and Manipulating Real Images

Now let's move from random noise to a real photograph. OpenCV's `cv2.imread()` loads an image from disk into a NumPy array — just like the ones we made above, but with millions of pixels instead of 100.

We can **slice** the array to crop a **Region of Interest (ROI)** — no special function needed, just standard NumPy indexing.

In [ ]:
IMAGE_PATH = "samples/images/kookaburra.jpeg"

# Load the image — OpenCV returns a NumPy array in BGR order
img = cv2.imread(IMAGE_PATH)
if img is None:
    raise FileNotFoundError(f"Could not load image at '{IMAGE_PATH}' — check the path exists.")

print(f"Original shape : {img.shape}  (H × W × C)")

# Resize so it fits comfortably on any screen
img_resized = cv2.resize(img, (640, 480))
print(f"Resized shape  : {img_resized.shape}")

# Crop a Region of Interest (ROI) using plain NumPy slicing: [rows, columns]
roi = img_resized[100:300, 200:400]
print(f"ROI shape      : {roi.shape}  (200 × 200 crop from the centre)")

# Display both windows side by side — press any key to close
cv2.namedWindow("Original (resized)")
cv2.namedWindow("Cropped ROI")
cv2.moveWindow("Original (resized)", 50, 100)
cv2.moveWindow("Cropped ROI", 720, 100)

cv2.imshow("Original (resized)", img_resized)
cv2.imshow("Cropped ROI", roi)

print("\nPress any key in the OpenCV window to continue...")
cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(1)  # macOS needs an extra pump to close the windows

## Colour Spaces: Why Gray Matters

OpenCV loads images in **BGR** (Blue, Green, Red) by default — not RGB! When working with other libraries (Matplotlib, SAM, CLIP), you'll need to convert.

**Grayscale** collapses 3 channels into 1, reducing data by 3×. Most traditional CV algorithms (edge detection, thresholding) work on grayscale because they only care about intensity, not colour.

**HSV** (Hue, Saturation, Value) separates *colour* from *brightness*, making it much easier to detect objects by colour under varying lighting conditions.

In [ ]:
# Convert from BGR → Grayscale and BGR → HSV
img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
img_hsv  = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)

# For Matplotlib display, also convert BGR → RGB (Matplotlib expects RGB order)
img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

print(f"BGR  shape : {img_resized.shape}   (3 channels)")
print(f"Gray shape : {img_gray.shape}        (1 channel — 3× less data!)")
print(f"HSV  shape : {img_hsv.shape}   (3 channels: Hue, Saturation, Value)")

# Show all three side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_rgb)
axes[0].set_title("Original (RGB)")

axes[1].imshow(img_gray, cmap="gray")
axes[1].set_title("Grayscale — 1 channel")

axes[2].imshow(img_hsv)
axes[2].set_title("HSV (raw channels as RGB)")

for ax in axes:
    ax.axis("off")

plt.suptitle("Colour Space Conversions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Edge Detection: Teaching the Computer to See Outlines

Edges are where pixel intensity changes sharply — they reveal the structure of a scene. The classic pipeline is:

1. **Gaussian Blur** — smooths out noise so the detector doesn't fire on every speck.
2. **Canny Edge Detection** — finds gradients (rapid changes) and traces them into clean edges.

The two threshold values in `cv2.Canny(image, low, high)` control sensitivity: pixels with gradients above `high` are definite edges; those between `low` and `high` are kept only if they connect to a strong edge.

In [ ]:
# Step 1 — Blur to reduce noise (kernel size must be odd, e.g. 7×7)
img_blur = cv2.GaussianBlur(img_gray, (7, 7), 0)

# Step 2 — Canny edge detection (low=100, high=200 thresholds)
img_canny = cv2.Canny(img_blur, 100, 200)

print(f"Edge map shape: {img_canny.shape}  (same H × W, single channel, values 0 or 255)")

# Show the full pipeline: Original → Blurred → Edges
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_gray, cmap="gray")
axes[0].set_title("1. Grayscale")

axes[1].imshow(img_blur, cmap="gray")
axes[1].set_title("2. Gaussian Blur (7×7)")

axes[2].imshow(img_canny, cmap="gray")
axes[2].set_title("3. Canny Edges (100 / 200)")

for ax in axes:
    ax.axis("off")

plt.suptitle("Edge Detection Pipeline", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Live Canny: See Every Step on Your Webcam

The static image above is helpful, but it's hard to *feel* what each parameter does. The cell below runs the full Canny pipeline **live on your webcam** and shows all four stages side by side:

| Panel | What You See |
|---|---|
| **Original** | Raw colour frame from the webcam |
| **Grayscale** | Single-channel intensity (3× less data) |
| **Gaussian Blur** | Smoothed version — noise removed, details softened |
| **Canny Edges** | Final edge map — white lines on black |

Use the **trackbar sliders** to change the blur kernel size and Canny thresholds in real time. Notice how:
- A **larger blur kernel** removes more detail (fewer, cleaner edges).
- A **lower Canny threshold** picks up more edges (but more noise too).
- A **higher Canny threshold** keeps only the strongest edges.

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("Live Canny Pipeline — drag the sliders to experiment!")
print("Press 'q' to quit.\n")

# --- Trackbar window for live parameter tuning ---
cv2.namedWindow("Controls", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Controls", 400, 120)
cv2.moveWindow("Controls", 50, 50)  # top-left corner
cv2.createTrackbar("Blur Kernel", "Controls", 3, 15, lambda _: None)   # 0–15 → mapped to odd
cv2.createTrackbar("Canny Low",   "Controls", 50, 255, lambda _: None)
cv2.createTrackbar("Canny High",  "Controls", 150, 255, lambda _: None)

# Position the main display below the controls
cv2.namedWindow("Edge Detection Pipeline (Live)", cv2.WINDOW_NORMAL)
cv2.moveWindow("Edge Detection Pipeline (Live)", 50, 220)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)

    # Read current slider values
    blur_val  = cv2.getTrackbarPos("Blur Kernel", "Controls")
    canny_low = cv2.getTrackbarPos("Canny Low",   "Controls")
    canny_high = cv2.getTrackbarPos("Canny High",  "Controls")

    # Blur kernel must be odd and >= 1
    k = max(1, blur_val * 2 + 1)

    # --- The four stages of the pipeline ---
    # 1. Grayscale (3 channels → 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian Blur (smooths noise — larger k = smoother)
    blurred = cv2.GaussianBlur(gray, (k, k), 0)

    # 3. Canny Edge Detection (finds intensity gradients)
    edges = cv2.Canny(blurred, canny_low, canny_high)

    # Convert single-channel images to BGR so we can stack them with the colour frame
    gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    blur_bgr = cv2.cvtColor(blurred, cv2.COLOR_GRAY2BGR)
    edge_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # Label each panel
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(frame,    "Original",                    (10, 25), font, 0.7, (0, 255, 0), 2)
    cv2.putText(gray_bgr, "Grayscale",                   (10, 25), font, 0.7, (0, 255, 0), 2)
    cv2.putText(blur_bgr, f"Blur k={k}",                 (10, 25), font, 0.7, (0, 255, 0), 2)
    cv2.putText(edge_bgr, f"Canny {canny_low}/{canny_high}", (10, 25), font, 0.7, (0, 255, 0), 2)

    # Arrange into a 2×2 grid
    top_row = np.hstack([frame, gray_bgr])
    bot_row = np.hstack([blur_bgr, edge_bgr])
    grid = np.vstack([top_row, bot_row])

    # Scale down to fit on screen
    display = cv2.resize(grid, (grid.shape[1] // 2, grid.shape[0] // 2))
    cv2.imshow("Edge Detection Pipeline (Live)", display)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

---

## Live Video: Your Webcam as a Matrix Stream

A video is nothing more than a **loop of images** (frames). OpenCV's `VideoCapture` grabs one frame at a time inside a `while` loop — each frame is the same kind of NumPy array we've been working with.

### The Core Pattern
```python
cap = cv2.VideoCapture(0)       # 0 = default webcam
while True:
    ret, frame = cap.read()     # Grab one frame
    # ... do something with frame ...
    cv2.imshow("Window", frame) # Show it
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
```
You'll see this same pattern in every notebook from here on — YOLO, SAM, and beyond.

> **Press `q`** to close any webcam window safely.

In [ ]:
# 0 = default webcam, try 1 or 2 for an external USB camera
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam — is it connected?")

# Read one frame to report dimensions
ret, test_frame = cap.read()
if ret:
    h, w = test_frame.shape[:2]
    print(f"Webcam opened: {w}×{h}")

print("Press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip horizontally so it behaves like a mirror (more intuitive)
    frame = cv2.flip(frame, 1)

    cv2.imshow("My Interactive Mirror", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

## Real-Time Processing: Live Pencil Sketch

Now we combine everything: grab a frame, convert to gray, blur, run Canny, and invert — all inside the webcam loop. The result is a **live pencil-sketch filter** running at full frame rate.

This is the same pipeline used in apps like Snapchat and Instagram filters — just more layers on top.

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("Live Pencil Sketch — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)

    # 1. Grayscale — drop colour info
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian Blur — smooth out skin texture / noise
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    # 3. Canny — detect edges (lower thresholds = more detail)
    edges = cv2.Canny(blur, 50, 150)

    # 4. Invert — white paper, dark lines (255 − value)
    sketch = cv2.bitwise_not(edges)

    cv2.imshow("Pencil Sketch Live", sketch)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

## Drawing on Frames: HUD Overlay

OpenCV can draw shapes and text directly onto any frame. The key functions are:
- `cv2.drawMarker()` — crosshairs, diamonds, stars
- `cv2.putText()` — text labels
- `cv2.rectangle()`, `cv2.circle()`, `cv2.line()` — basic shapes

Below is a snippet showing how to add a targeting reticle.

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("HUD Overlay — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)

    h, w = frame.shape[:2]
    cx, cy = w // 2, h // 2  # dynamic centre based on actual frame size

    # Crosshair reticle at the centre
    cv2.drawMarker(frame, (cx, cy), (0, 0, 255),
                   markerType=cv2.MARKER_CROSS, markerSize=40, thickness=2)

    # Targeting circle
    cv2.circle(frame, (cx, cy), 80, (0, 0, 255), 1)

    # HUD text
    cv2.putText(frame, "TARGETING SYSTEM ACTIVE", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(frame, f"Centre: ({cx}, {cy})", (10, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.imshow("HUD Overlay", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

---

## Colour Filtering: Isolating Objects by Hue

Remember HSV? Here's where it shines. By defining a **range of Hue values**, we can create a binary mask that isolates only pixels of a specific colour — regardless of lighting.

This is the foundation of many real-world systems: detecting safety vests, tracking coloured markers, lane detection, and more.

### How It Works
1. Convert the frame to **HSV** colour space.
2. Define a lower and upper bound for the target colour.
3. `cv2.inRange()` creates a binary mask: white where the colour matches, black everywhere else.

In [ ]:
# --- Colour filter: isolate blue pixels in real time ---
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

# HSV range for blue (tweak these to match your object)
LOWER_BLUE = np.array([100, 150, 50])
UPPER_BLUE = np.array([140, 255, 255])

print("Blue Colour Filter — hold up something blue!")
print("Press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)

    # Convert to HSV so we can filter by hue
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Create binary mask: white where colour is in range, black elsewhere
    mask = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE)

    # Also show the masked colour feed for context
    colour_only = cv2.bitwise_and(frame, frame, mask=mask)

    cv2.imshow("Binary Mask (blue only)", mask)
    cv2.imshow("Colour Feed (blue isolated)", colour_only)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

### Challenge: Detect a Different Colour

Modify the HSV ranges to detect **red** or **green** objects instead of blue. Here are some starting ranges to experiment with:

| Colour | Lower HSV | Upper HSV |
|---|---|---|
| Blue | `[100, 150, 50]` | `[140, 255, 255]` |
| Green | `[35, 100, 50]` | `[85, 255, 255]` |
| Red (low) | `[0, 120, 70]` | `[10, 255, 255]` |
| Red (high) | `[170, 120, 70]` | `[180, 255, 255]` |

> **Tip:** Red wraps around the hue wheel, so you need two ranges combined with `cv2.bitwise_or()`.

---

## Putting It Together: Object Tracking Drawing App

Now we combine colour filtering with **contour detection** to track a coloured object and draw a trail as it moves — a miniature augmented reality app built entirely with OpenCV.

In [ ]:
# --- Drawing App: track a blue object and leave a coloured trail ---
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

# HSV range for blue — reuse the same values from above
LOWER_BLUE = np.array([100, 150, 50])
UPPER_BLUE = np.array([140, 255, 255])

MIN_RADIUS = 10  # ignore detections smaller than this (noise)

# Create a blank canvas the same size as the webcam frame for the trail
ret, first_frame = cap.read()
if not ret:
    raise RuntimeError("Could not read initial frame.")
trail = np.zeros_like(first_frame)

print("Drawing App — wave a blue object to draw!")
print("Press 'c' to clear the canvas, 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)

    # Convert to HSV and create a mask for the target colour
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE)

    # Find contours in the mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Pick the largest contour (most likely the real object, not noise)
        largest = max(contours, key=cv2.contourArea)
        (x, y), radius = cv2.minEnclosingCircle(largest)

        if radius > MIN_RADIUS:
            center = (int(x), int(y))
            # Live indicator on the camera feed
            cv2.circle(frame, center, int(radius), (255, 0, 0), 2)
            # Permanent dot on the trail canvas
            cv2.circle(trail, center, 8, (255, 0, 0), -1)

    # Blend the trail onto the live feed
    output = cv2.addWeighted(frame, 0.7, trail, 0.7, 0)
    cv2.imshow("Drawing Trail", output)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break
    elif key == ord("c"):
        trail[:] = 0  # clear the canvas

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

### Experiment
- Change the colour range to track a different object.
- Adjust the trail thickness (`8`) or colour `(255, 0, 0)`.
- Press `q` and restart to clear the canvas.

---

## Recap

| What We Did | Key Function / Concept |
|---|---|
| **Images as matrices** | `np.random.randint()` — images are just NumPy arrays |
| **Load & crop** | `cv2.imread()`, array slicing `[y1:y2, x1:x2]` |
| **Colour spaces** | `cv2.cvtColor()` — BGR ↔ Gray ↔ HSV |
| **Edge detection** | `cv2.GaussianBlur()` + `cv2.Canny()` |
| **Live webcam** | `cv2.VideoCapture(0)` + `while` loop |
| **Drawing & text** | `cv2.putText()`, `cv2.drawMarker()`, `cv2.circle()` |
| **Colour filtering** | `cv2.inRange()` in HSV space |
| **Object tracking** | `cv2.findContours()` + `cv2.minEnclosingCircle()` |

### What's Next?
In Notebook 02, we hand over the hard work to AI. **YOLO** will detect, classify, and track objects in real time — using the same webcam loop you just learned here. Everything from this notebook is the foundation that YOLO builds on.